# Shared-fold cross-validation for Experiments

Models:
- Logistic Regression
- SVC
- QAE + SVC
- QAE + Q OCSVM

This notebook makes all models use the **exact same outer CV splits**. Preprocessing is fit **inside each fold** to avoid leakage.

In [28]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "preprocessing_cv_experiments.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# print("Project root:", PROJECT_ROOT)

In [ ]:

import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.svm import OneClassSVM, SVC
from sklearn.linear_model import LogisticRegression
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed


from preprocessing_cv_experiments import (
    load_kdd99,
    load_cic_iot23,
    kdd99_binary_label_map,
    benign_binary_label_map,
    make_fixed_stratified_subset,
    make_shared_kfolds,
    FoldPreprocessorConfig,
    FoldPreprocessor,
    run_linear_pca2_on_shared_folds,
    run_svm_pca4_on_shared_folds,
    summarize_metrics,
    compute_metrics,
    make_inner_validation_split,
)


from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score,
    f1_score, 
    roc_auc_score,
)


from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, OneClassSVM

from qae import QAEConfig, QuantumAutoencoder


# To paralelize fold computattions
from joblib import Parallel, delayed
import os
os.environ["OMP_NUM_THREADS"] = "10"
os.environ["MKL_NUM_THREADS"] = "10"
os.environ["OPENBLAS_NUM_THREADS"] = "10"


In [ ]:
# Experiment config
DATASET = "kdd99"   

KDD_PERCENT10 = True

CIC_PATH = PROJECT_ROOT / "data" / "Merged01.csv"
CIC_LABEL_COL = "Label"
CIC_BENIGN_VALUES = ("benign",)

SUBSET_SIZE = 15000
N_SPLITS = 5
CV_SEED = 42

PREPROC_CONFIG = FoldPreprocessorConfig(
    n_components=4,
    categorical_cols=None,
    drop_cols=None,
    clip_value=1e12,
    balance_train=True,
    random_state=42,
)

QAE_CONFIG = QAEConfig(
    n_qubits=4,
    n_latent=2,
    n_layers=1,
    device_name="default.qubit",
    seed=42,
    use_swap_test_loss=True,
    use_log_cost=True,
)

QAE_TRAIN_STEPS = 40
QAE_LR = 0.03
QAE_BATCH_SIZE = 32
QAE_VAL_FRACTION = 0.2

MAX_QAE_TRAIN_SAMPLES = 6000
MAX_QAE_VAL_SAMPLES = 1000

MAX_QAE_SVM_TRAIN_SAMPLES = None
MAX_QAE_SVM_TEST_SAMPLES = None

rng_global = np.random.default_rng(42)

In [31]:
if DATASET == "kdd99":
    df = load_kdd99(percent10=KDD_PERCENT10)
    label_col = "labels"
    label_map_fn = kdd99_binary_label_map
elif DATASET == "cic_iot23":
    df = load_cic_iot23(CIC_PATH)
    label_col = CIC_LABEL_COL
    label_map_fn = benign_binary_label_map(CIC_BENIGN_VALUES)
else:
    raise ValueError(f"Unknown DATASET: {DATASET}")

df = df.copy()
df[label_col] = label_map_fn(df[label_col]).astype(int)

X_df_full = df.drop(columns=[label_col])
y_full = df[label_col].to_numpy()

X_df, y = make_fixed_stratified_subset(
    X_df_full,
    y_full,
    total_size=SUBSET_SIZE,
    random_state=CV_SEED,
)

print("Dataset:", DATASET)
print("Raw full shape:", X_df_full.shape)
print("Fixed pooled subset shape:", X_df.shape)
print("Subset class counts:", pd.Series(y).value_counts().to_dict())

Dataset: kdd99
Raw full shape: (494021, 41)
Fixed pooled subset shape: (15000, 41)
Subset class counts: {1: 12046, 0: 2954}


In [32]:
folds = make_shared_kfolds(
    y=y,
    n_splits=N_SPLITS,
    random_state=CV_SEED,
    shuffle=True,
)

print(f"Total outer folds: {len(folds)}")
print("Example fold sizes:", len(folds[0][1]), len(folds[0][2]))

Total outer folds: 5
Example fold sizes: 12000 3000


## Classical baselines on shared folds

In [6]:

linear_results = run_linear_pca2_on_shared_folds(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
)

svm_results = run_svm_pca4_on_shared_folds(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
)

classical_results = pd.concat([linear_results, svm_results], ignore_index=True)
classical_results.head()


,fold,model,accuracy,precision,recall,f1,roc_auc
0,1,PCA-2 + LogisticRegression,0.970667,0.982143,0.981328,0.981735,0.993327
1,2,PCA-2 + LogisticRegression,0.976000,0.989527,0.980490,0.984987,0.993786
2,3,PCA-2 + LogisticRegression,0.975667,0.989933,0.979660,0.984769,0.993773
3,4,PCA-2 + LogisticRegression,0.977000,0.989540,0.981735,0.985622,0.993668
4,5,PCA-2 + LogisticRegression,0.978333,0.992851,0.980075,0.986422,0.995934


In [7]:

classical_summary = (
    classical_results
    .groupby("model", as_index=False)
    .apply(lambda g: summarize_metrics(g))
    .reset_index(level=0)
    .rename(columns={"level_0": "group_idx"})
)

for model_name in classical_results["model"].unique():
    print("\n", model_name)
    display(summarize_metrics(classical_results[classical_results["model"] == model_name])[["metric", "mean ± std"]])



 PCA-2 + LogisticRegression


,metric,mean ± std
0,accuracy,0.9755 ± 0.0029
1,precision,0.9888 ± 0.0040
2,recall,0.9807 ± 0.0009
3,f1,0.9847 ± 0.0018
4,roc_auc,0.9941 ± 0.0010



 PCA-4 + SVM


,metric,mean ± std
0,accuracy,0.9857 ± 0.0017
1,precision,1.0000 ± 0.0000
2,recall,0.9822 ± 0.0021
3,f1,0.9910 ± 0.0011
4,roc_auc,0.9942 ± 0.0015


## Model comparison

In [ ]:
def evaluate_kernel_batched(qkernel, X, Y=None, batch_size=25, desc="Kernel"):
    if Y is None:
        Y = X

    blocks = []
    for start in tqdm(range(0, len(X), batch_size), desc=desc):
        end = min(start + batch_size, len(X))
        K_chunk = qkernel.evaluate(x_vec=X[start:end], y_vec=Y)
        blocks.append(K_chunk)

    return np.vstack(blocks)


def subsample_balanced_by_index(y, n_samples, rng):
    y = np.asarray(y)
    classes = np.unique(y)

    if n_samples is None or n_samples >= len(y):
        idx = np.arange(len(y))
        rng.shuffle(idx)
        return idx

    n_per_class = n_samples // len(classes)
    indices = []

    for c in classes:
        c_idx = np.where(y == c)[0]
        chosen = rng.choice(c_idx, size=min(n_per_class, len(c_idx)), replace=False)
        indices.append(chosen)

    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return indices


def _compute_metrics(name, fold_id, yt, yp, yprob, elapsed):
    return {
        "fold": fold_id + 1,
        "model": name,
        "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "roc_auc": (
            roc_auc_score(yt, yprob)
            if yprob is not None and len(np.unique(yt)) > 1
            else np.nan
        ),
        "time_sec": elapsed,
    }


def _run_one_fold(
    X_df,
    y,
    fold,
    preproc_config,
    qae_config,
    qae_steps,
    qae_lr,
    qae_batch_size,
    qae_val_fraction,
    qsvm_reps,
    qsvm_nu,
    qsvm_kernel_batch,
    random_state,
    n_train=None,
    n_test=None,
    max_ocsvm_normals=None,
):
    fold_id, train_idx, test_idx = fold
    fold_seed = 1000 + fold_id + random_state
    rng = np.random.default_rng(fold_seed)

    fp = FoldPreprocessor(
        FoldPreprocessorConfig(
            n_components=preproc_config.n_components,
            categorical_cols=preproc_config.categorical_cols,
            drop_cols=preproc_config.drop_cols,
            clip_value=preproc_config.clip_value,
            balance_train=preproc_config.balance_train,
            random_state=fold_seed,
        )
    )

    train_pack = fp.fit_transform(X_df.iloc[train_idx], y[train_idx])
    test_pack = fp.transform(X_df.iloc[test_idx])

    X_train_pca = train_pack["X_train_pca"]
    y_train_full = train_pack["y_train"]
    X_test_pca = test_pack["X_test_pca"]
    y_test_full = y[test_idx]

    train_sel = subsample_balanced_by_index(y_train_full, n_train, rng)
    test_sel = subsample_balanced_by_index(y_test_full, n_test, rng)

    X_tr = X_train_pca[train_sel]
    y_tr = y_train_full[train_sel]
    X_te = X_test_pca[test_sel]
    y_te = y_test_full[test_sel]

    rows = []

    # 1) Logistic Regression
    t0 = time.time()
    lr_clf = LogisticRegression(max_iter=1000, random_state=fold_seed)
    lr_clf.fit(X_tr, y_tr)
    lr_pred = lr_clf.predict(X_te)
    lr_prob = lr_clf.predict_proba(X_te)[:, 1]
    rows.append(_compute_metrics("Logistic Regression (PCA)", fold_id, y_te, lr_pred, lr_prob, time.time() - t0))

    # 2) Classical SVM
    t0 = time.time()
    svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=fold_seed)
    svm_clf.fit(X_tr, y_tr)
    svm_pred = svm_clf.predict(X_te)
    svm_prob = svm_clf.predict_proba(X_te)[:, 1]
    rows.append(_compute_metrics("Classical SVM (PCA)", fold_id, y_te, svm_pred, svm_prob, time.time() - t0))

    # 3) QAE + Classical SVM
    t0 = time.time()

    inner_tr, inner_val = train_test_split(
        np.arange(len(X_tr)),
        test_size=qae_val_fraction,
        random_state=fold_seed,
        stratify=y_tr,
    )

    X_qae_tr = X_tr[inner_tr]
    X_qae_val = X_tr[inner_val]

    qae_cfg = QAEConfig(
        n_qubits=qae_config.n_qubits,
        n_latent=qae_config.n_latent,
        n_layers=qae_config.n_layers,
        device_name=qae_config.device_name,
        seed=fold_seed,
        use_swap_test_loss=qae_config.use_swap_test_loss,
        use_log_cost=qae_config.use_log_cost,
        eps=qae_config.eps,
    )
    qae = QuantumAutoencoder(qae_cfg)

    qae_best_params, qae_history, qae_scaler = qae.train(
        X_train_raw=X_qae_tr,
        X_val_raw=X_qae_val,
        steps=qae_steps,
        lr=qae_lr,
        batch_size=qae_batch_size,
        val_every=max(5, qae_steps // 8),
        verbose=False,
    )

    Z_tr = qae.batch_transform_latent_features_from_raw(
        X_tr, qae_best_params, qae_scaler, batch_size=1000, verbose=False
    )
    Z_te = qae.batch_transform_latent_features_from_raw(
        X_te, qae_best_params, qae_scaler, batch_size=1000, verbose=False
    )

    qae_svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=fold_seed)
    qae_svm_clf.fit(Z_tr, y_tr)
    qae_svm_pred = qae_svm_clf.predict(Z_te)
    qae_svm_prob = qae_svm_clf.predict_proba(Z_te)[:, 1]
    rows.append(_compute_metrics("QAE + Classical SVM", fold_id, y_te, qae_svm_pred, qae_svm_prob, time.time() - t0))

    # QAE + OCSVM
    t0 = time.time()

    attack_mask = (y_tr == 1)
    Z_train_ocsvm = Z_tr[attack_mask]

    if max_ocsvm_normals is not None and len(Z_train_ocsvm) > max_ocsvm_normals:
        chosen = rng.choice(len(Z_train_ocsvm), size=max_ocsvm_normals, replace=False)
        Z_train_ocsvm = Z_train_ocsvm[chosen]

    feature_dim = Z_tr.shape[1]
    fm = zz_feature_map(feature_dimension=feature_dim, reps=qsvm_reps)
    qkernel = FidelityQuantumKernel(feature_map=fm)

    K_train = evaluate_kernel_batched(
        qkernel,
        Z_train_ocsvm,
        Y=Z_train_ocsvm,
        batch_size=qsvm_kernel_batch,
        desc=f"QK train f{fold_id + 1}",
    )
    K_test = evaluate_kernel_batched(
        qkernel,
        Z_te,
        Y=Z_train_ocsvm,
        batch_size=qsvm_kernel_batch,
        desc=f"QK test f{fold_id + 1}",
    )

    ocsvm = OneClassSVM(kernel="precomputed", nu=qsvm_nu)
    ocsvm.fit(K_train)
    ocsvm_raw = ocsvm.predict(K_test)
    ocsvm_pred = np.where(ocsvm_raw == 1, 1, 0)

    rows.append(_compute_metrics("QAE + Quantum OC-SVM", fold_id, y_te, ocsvm_pred, None, time.time() - t0))

    artifacts = {
        "qae": qae,
        "qae_best_params": qae_best_params,
        "qae_scaler": qae_scaler,
        "qae_history": qae_history,
        "X_tr_pca": X_tr,
        "X_te_pca": X_te,
        "Z_tr": Z_tr,
        "Z_te": Z_te,
        "y_tr": y_tr,
        "y_te": y_te,
    }

    return rows, artifacts


def run_4model_cv(
    X_df,
    y,
    folds,
    preproc_config,
    qae_config,
    qae_steps=40,
    qae_lr=0.03,
    qae_batch_size=32,
    qae_val_fraction=0.2,
    qsvm_reps=1,
    qsvm_nu=0.1,
    qsvm_kernel_batch=20,
    random_state=42,
    n_jobs=1,
    n_train=None,
    n_test=None,
    max_ocsvm_normals=None,
):
    # Run folds on separate CPUs 
    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_run_one_fold)(
            X_df=X_df,
            y=y,
            fold=fold,
            preproc_config=preproc_config,
            qae_config=qae_config,
            qae_steps=qae_steps,
            qae_lr=qae_lr,
            qae_batch_size=qae_batch_size,
            qae_val_fraction=qae_val_fraction,
            qsvm_reps=qsvm_reps,
            qsvm_nu=qsvm_nu,
            qsvm_kernel_batch=qsvm_kernel_batch,
            random_state=random_state,
            n_train=n_train,
            n_test=n_test,
            max_ocsvm_normals=max_ocsvm_normals,
        )
        for fold in folds
    )

    all_rows = []
    last_artifacts = None

    for rows, artifacts in results:
        all_rows.extend(rows)
        last_artifacts = artifacts

    per_fold_df = pd.DataFrame(all_rows)

    metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "time_sec"]
    summary_rows = []

    for model_name in per_fold_df["model"].unique():
        mdf = per_fold_df[per_fold_df["model"] == model_name]
        row = {"model": model_name}
        for m in metric_cols:
            mean = mdf[m].mean()
            std = mdf[m].std(ddof=1)
            row[f"{m}_mean"] = mean
            row[f"{m}_std"] = std
            row[m] = f"{mean:.4f} ± {std:.4f}"
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    return per_fold_df, summary_df, last_artifacts

In [ ]:
per_fold_df, summary_df, last_artifacts = run_4model_cv(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
    qae_config=QAE_CONFIG,

    qae_steps=50,
    qae_lr=0.03,
    qae_batch_size=64,
    qae_val_fraction=0.2,

    qsvm_reps=2,             
    qsvm_nu=0.05,
    qsvm_kernel_batch=100,

    n_train=None, # None == take all
    n_test=None, # None == take all
    max_ocsvm_normals=200,     
    random_state=42,
    n_jobs=5,
)

QK test f1: 100%|██████████| 30/30 [24:37<00:00, 49.26s/it]


In [50]:
display(
    summary_df[["model", "accuracy", "precision", "recall", "f1"]]
    .style.hide(axis="index")
)
print("\nPer-fold metrics:")
display(
    per_fold_df
    .style.format({
        "accuracy": "{:.4f}", "precision": "{:.4f}",
        "recall": "{:.4f}", "f1": "{:.4f}",
        "roc_auc": "{:.4f}", "time_sec": "{:.2f}",
    })
    .hide(axis="index")
)

model,accuracy,precision,recall,f1
Logistic Regression (PCA),0.9844 ± 0.0018,0.9985 ± 0.0010,0.9821 ± 0.0021,0.9902 ± 0.0011
Classical SVM (PCA),0.9856 ± 0.0018,1.0000 ± 0.0000,0.9821 ± 0.0023,0.9910 ± 0.0012
QAE + Classical SVM,0.9407 ± 0.0049,0.9996 ± 0.0003,0.9266 ± 0.0059,0.9617 ± 0.0033
QAE + Quantum OC-SVM,0.8438 ± 0.2182,0.9421 ± 0.0562,0.8463 ± 0.2532,0.8779 ± 0.1910



Per-fold metrics:


fold,model,accuracy,precision,recall,f1,roc_auc,time_sec
1,Logistic Regression (PCA),0.9843,0.9975,0.9830,0.9902,0.9959,0.03
1,Classical SVM (PCA),0.9863,1.0000,0.9830,0.9914,0.9963,0.18
1,QAE + Classical SVM,0.9397,0.9991,0.9257,0.9610,0.9886,142.08
1,QAE + Quantum OC-SVM,0.9360,0.9664,0.9535,0.9599,nan,1574.97
2,Logistic Regression (PCA),0.9833,0.9975,0.9817,0.9895,0.9956,0.06
2,Classical SVM (PCA),0.9857,1.0000,0.9822,0.9910,0.9937,0.15
2,QAE + Classical SVM,0.9443,0.9996,0.9311,0.9641,0.9864,141.84
2,QAE + Quantum OC-SVM,0.4537,0.8419,0.3935,0.5364,nan,1553.28
3,Logistic Regression (PCA),0.9827,0.9987,0.9797,0.9891,0.9950,0.05
3,Classical SVM (PCA),0.9830,1.0000,0.9788,0.9893,0.9902,0.22
